# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Scepter70/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Scepter70/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Type: Scoring, used to produce a ranking.**

This is a scoring task, not a hard classification. The deliverable a reviewer actually uses isn't a single accept/reject label per page — it's an ordered list of the top N pages to look at first. I train a model to output a probability/score per page (how likely this page is a good refresh candidate), then rank all eligible pages by that score. Classification tools (e.g. a decision tree or random forest classifier) are the mechanism, but the *task* is scoring-for-ranking, because what matters operationally is the ordering of the top 50, not whether any single page crosses a 0.5 threshold.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target/proxy:** whether a page's `trend_direction` is `"down"` over the observed 90-day window in the starter dataset.

This label comes from an **observed outcome** in the historical data, not a rule I invented myself — `trend_direction` reflects how the page's traffic actually moved, computed from real (anonymized) impressions/clicks history. I'm using it as a *proxy* for "needs a refresh," because a real ground-truth label (did refreshing this page actually help?) doesn't exist in this dataset — that would require an intervention experiment. So the honest framing is: I'm predicting an observed decline signal, and treating decline as a reasonable proxy for refresh-worthiness, not a proven causal target.

In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]

print(df["trend_direction"].value_counts())
print(f"\nProportion trending down (my proxy label): {(df['trend_direction']=='down').mean():.1%}")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Proportion trending down (my proxy label): 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.**

Since a reviewer only ever looks at a fixed-size top slice of the ranking (e.g. the top 50 pages each cycle), overall accuracy across all pages is the wrong metric — it rewards being right about pages nobody will ever look at. Precision@50 asks the one question that matters operationally: of the top 50 pages my ranking surfaces, what fraction are actually genuine candidates (trend_direction = down)? That's a number I can defend directly to a reviewer, because it maps one-to-one onto their actual workflow and limited review capacity.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page**, observed over its trailing 90-day window, after filtering to pages that both received impressions and are old enough to have a meaningful trend (content_age_days >= 90).

In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

# Sketch the target column I'd train on
df["target_needs_refresh"] = (df["trend_direction"] == "down").astype(int)

print(f"Rows (= pages) in this slice: {len(df):,}")
print(f"One row = one page, observed over its trailing 90-day window.\n")
cols_to_show = [c for c in ["impressions_90d", "content_age_days", "trend_direction", "target_needs_refresh"] if c in df.columns]
print(df[cols_to_show].head(5))


Rows (= pages) in this slice: 30,000
One row = one page, observed over its trailing 90-day window.

   impressions_90d  content_age_days trend_direction  target_needs_refresh
0             3803               187            down                     1
1            15320               445            down                     1
2            12581               141            down                     1
3            11751               463          stable                     0
4            19140               263            down                     1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The starter pipeline already tested a hand-written rule against a trained model on this exact lane, and the gap is the evidence: the baseline rule scores **0.240 precision@50**, the random forest scores **0.740 precision@50** (`outputs/model_report.md`) — roughly three times as many genuine candidates in the top 50.

That gap exists because "needs a refresh" isn't governed by any single threshold — it's an interaction of several signals at once (impressions trend, position movement, content age, engagement) where the *combination* matters more than any one variable crossing a line. A fixed if-statement rule can only check one or two conditions cleanly; it can't learn that, say, moderate traffic decline plus high content age plus falling position together is a much stronger signal than any one of those alone. That's exactly the kind of multi-variable, non-linear pattern a model can pick up on and a fixed rule can't.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.